<<div style="text-align: center; padding: 20px; background-color: #F4F6F6; border-radius: 10px;">
    <h1 style="color: #2E86AB; margin-bottom: 5px;"> E-Commerce Sales Analysis 🛒</h1>
    <h3 style="color: #555; margin-top: 0;">Using SQL & SQLite</h3>
    <hr style="border: 1px solid #2E86AB; width: 50%;">
    <p style="color: #333;"><b>Author:</b> Aryam Khaled | <b>Date:</b> November 2026</p>
    <p style="color: #333;"><b>Tools:</b> Python, Pandas, SQLite, SQL</p>
</div>

<br>

<div style="background-color: #f5f7fa; padding: 15px; border-left: 5px solid #2E86AB; border-radius: 5px;">
    <b style="color: #23465f; font-size: 18px;">Table of Contents</b>
    <ol style="color: #333; margin-top: 10px; line-height: 1.8;">
        <li><a href="#section1" style="color: #2E86AB;"><b>Database Setup</b></a> (Creating Tables & Clean View)</li>
        <li><a href="#section2" style="color: #2E86AB;"><b>Sales & Products Analysis</b></a> (Top Products by Quantity & Revenue)</li>
        <li><a href="#section3" style="color: #2E86AB;"><b>Customers & Geographic Analysis</b></a> (Top Customers & Sales by Country)</li>
    </ol>
</div>

In [1]:
import pandas as pd
import openpyxl

In [2]:
file_path= r'../data/Online Retail.xlsx'
df_Ret=pd.read_excel(file_path)

In [3]:
df_Ret.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df_products=df_Ret[['StockCode','Description','UnitPrice']]
df_products

,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
...,...,...,...
541904,22613,PACK OF 20 SPACEBOY NAPKINS,0.85
541905,22899,CHILDREN'S APRON DOLLY GIRL,2.10
541906,23254,CHILDRENS CUTLERY DOLLY GIRL,4.15
541907,23255,CHILDRENS CUTLERY CIRCUS PARADE,4.15


In [5]:
df_products= df_products.drop_duplicates(subset=['StockCode'])
df_products.shape

(4070, 3)

In [6]:
df_invoices= df_Ret[['InvoiceNo','InvoiceDate','CustomerID','Country']]
df_invoices

,InvoiceNo,InvoiceDate,CustomerID,Country
0,536365,2010-12-01 08:26:00,17850.0,United Kingdom
1,536365,2010-12-01 08:26:00,17850.0,United Kingdom
2,536365,2010-12-01 08:26:00,17850.0,United Kingdom
3,536365,2010-12-01 08:26:00,17850.0,United Kingdom
4,536365,2010-12-01 08:26:00,17850.0,United Kingdom
...,...,...,...,...
541904,581587,2011-12-09 12:50:00,12680.0,France
541905,581587,2011-12-09 12:50:00,12680.0,France
541906,581587,2011-12-09 12:50:00,12680.0,France
541907,581587,2011-12-09 12:50:00,12680.0,France


In [7]:
df_invoices= df_invoices.drop_duplicates(subset=['InvoiceNo'])
df_invoices.shape

(25900, 4)

In [8]:
df_invoice_details= df_Ret[['InvoiceNo','StockCode','Quantity']]
df_invoice_details.shape

(541909, 3)

<h2 id="section1">1. Database Setup</h2>

In this section, we create a SQLite database and load the data into three separate tables: Products, Invoices, and Invoice_Details.

In [9]:
import sqlite3

conn= sqlite3.connect('../data/retail.db')

df_products.to_sql('Products',conn,if_exists='replace',index=False)
df_invoices.to_sql('Invoices',conn,if_exists='replace',index=False)
df_invoice_details.to_sql('Invoice_Details',conn,if_exists='replace',index=False)



541909

In [10]:
query=" SELECT name FROM sqlite_master WHERE type='table';"
tables=pd.read_sql(query,conn)

print(tables)

#conn.close()


              name
0         Products
1         Invoices
2  Invoice_Details


In [11]:
# Create a clean view excluding administrative items
create_view_query = """
CREATE VIEW IF NOT EXISTS Clean_Sales AS
SELECT 
    d.InvoiceNo,
    d.StockCode,
    d.Quantity,
    p.Description,
    p.UnitPrice
FROM Invoice_Details AS d
JOIN Products AS p ON d.StockCode = p.StockCode
WHERE p.Description NOT IN (
    'POSTAGE', 'DOTCOM POSTAGE', 'Manual', 'Discount', 'mailout', 
    'BANK CHARGES', 'AMAZON FEE', 'CRUK Commission', 'Adjust bad debt'
);
"""

# Execute the view creation
cursor = conn.cursor()
cursor.execute(create_view_query)
conn.commit() #important
print("View 'Clean_Sales' created successfully! ✅ ")

View 'Clean_Sales' created successfully! ✅ 


<h2 id="section2">2. Sales & Products Analysis</h2>


### Task 1: Top 5 Products by Quantity
*Goal: Identify the most sold products (using StockCode).*

In [17]:
query= """
SELECT StocKCode ,SUM(Quantity) as TotalQuantity 
FROM Clean_Sales 
GROUP BY StockCode
ORDER BY TotalQuantity DESC
LIMIT 5 ;
"""

top_products= pd.read_sql(query,conn)
print(top_products)

  StockCode  TotalQuantity
0     22197          56450
1     84077          53847
2    85099B          47363
3    85123A          38830
4     84879          36221


### Task 2: Top 5 Actual Products 


In [13]:
query="""
SELECT Description , SUM(Quantity) as TotalQuantity
FROM Clean_Sales
GROUP BY Description
ORDER BY TotalQuantity DESC
LIMIT 5;

"""

top_products_name= pd.read_sql(query,conn)
print(top_products_name)

                          Description  TotalQuantity
0                SMALL POPCORN HOLDER          56450
1   WORLD WAR 2 GLIDERS ASSTD DESIGNS          53847
2             JUMBO BAG RED RETROSPOT          47363
3  WHITE HANGING HEART T-LIGHT HOLDER          39122
4       ASSORTED COLOUR BIRD ORNAMENT          36221


### Task 3: Top 5 Products by Revenue
*Goal: Calculate total revenue (Quantity × UnitPrice).*

In [14]:
query= """
SELECT Description , SUM(Quantity * UnitPrice) as TotalRevenue
FROM Clean_Sales
GROUP BY Description
ORDER BY TotalRevenue DESC
LIMIT 5;

"""

top_product_revenue=pd.read_sql(query,conn)
print(top_product_revenue)

                          Description  TotalRevenue
0                       PARTY BUNTING     167965.04
1            REGENCY CAKESTAND 3 TIER     142131.00
2  WHITE HANGING HEART T-LIGHT HOLDER     100993.34
3             JUMBO BAG RED RETROSPOT      78148.95
4                  RABBIT NIGHT LIGHT      63743.68


<h2 id="section3">3. Customers & Geographic Analysis</h2>

### Task 1: Top 5 Customers by Spending

*Goal: Identify the most valuable customers based on their total spending.*

In [15]:

query="""
SELECT i.CustomerID , SUM(c.Quantity * c.UnitPrice) as TotalSpent
FROM Invoices AS i 
JOIN Clean_Sales AS c 
ON i.InvoiceNo = c.InvoiceNo
WHERE i.CustomerID IS NOT NULL
GROUP BY i.CustomerID 
ORDER BY TotalSpent DESC
LIMIT 5;
"""
top_customer_spent=pd.read_sql(query,conn)
print(top_customer_spent)

   CustomerID  TotalSpent
0     18102.0   408759.96
1     14646.0   355711.65
2     17450.0   186171.53
3     14911.0   174223.37
4     12415.0   155116.04


### Task 2: Sales by Country

In [16]:
query="""
SELECT i.Country ,SUM(c.Quantity * c.UnitPrice) as SalesbyCountry
FROM Invoices AS i
JOIN Clean_Sales AS c 
ON i.InvoiceNo = c.InvoiceNo
GROUP BY i.Country 
ORDER BY SalesbyCountry DESC
LIMIT 5;
"""
Sales_by_Country=pd.read_sql(query,conn)
print(Sales_by_Country)

          Country  SalesbyCountry
0  United Kingdom     9013533.594
1     Netherlands      361449.200
2            EIRE      333987.480
3         Germany      238660.660
4          France      210618.480


<br>
<hr style="border: 1px solid #ddd;">
<div style="text-align: center; color: #555; padding: 10px;">
    <h3> Analysis Complete </h3>
    <p>This notebook demonstrates a complete SQL-based analysis of e-commerce data, from database setup to advanced queries using JOINs and Views.</p>
    <p><b>Author:</b> Aryam Khaled | <b>Date:</b> November 2026</p>
    <p><b>Repository:</b> <a href="https://github.com/aryamkhaled/ecommerce-sql-analysis">github.com/aryamkhaled/ecommerce-sql-analysis</a></p>
</div>